# OpenAQ Data Source Discovery and Validation

Evaluating whether OpenAQ can provide suitable air-quality data for the Karachi AQI forecasting project.

The notebook performs the following steps:

1. Loads configuration and environment variables
2. Discovers monitoring locations near Karachi
3. Identifies relevant pollutant sensors
4. Downloads a recent sample of hourly observations
5. Evaluates completeness, freshness, missing values, and duplicates
6. Determines whether each sensor is suitable for:
   - historical model training
   - live AQI ingestion

### Imports

In [5]:
from __future__ import annotations

import json
import os
import sys
import time
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from pathlib import Path
from typing import Any

import pandas as pd
import requests
from dotenv import load_dotenv

## 1. Project and API Configuration

In this section we will define:

- the OpenAQ API base URL
- Karachi's reference coordinates
- the sensor search radius
- the number of historical days to test
- pagination limits
- required pollutants
- sensor acceptance thresholds

### Project Root and Environment Variables

In [6]:
load_dotenv()

API_KEY = os.getenv("OPENAQ_API_KEY")

if not API_KEY:
    raise RuntimeError(
        "OPENAQ_API_KEY is missing. Add it to the project's .env file."
    )


PROJECT_ROOT = Path.cwd().parent
RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
REPORTS_DIR = PROJECT_ROOT / "data" / "reports"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

### OpenAQ Settings

In [7]:
BASE_URL = "https://api.openaq.org/v3"

KARACHI_LATITUDE = 24.8607
KARACHI_LONGITUDE = 67.0011

# OpenAQ supports a maximum search radius of 25,000 metres.
SEARCH_RADIUS_METRES = 25_000

# Recent history used for initial feasibility testing.
SAMPLE_DAYS = 30

# Maximum number of records requested per page.
PAGE_LIMIT = 1_000

OUTPUT_DIR = REPORTS_DIR / "openaq_feasibility"

REQUIRED_POLLUTANTS = {
    "pm25",
    "pm10",
    "no2",
    "so2",
    "co",
    "o3",
}

# PM2.5 is the minimum pollutant required for this forecasting project.
MINIMUM_TARGET_POLLUTANTS = {"pm25"}

MAX_LIVE_DATA_AGE_HOURS = 6

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Reports will be saved to: {OUTPUT_DIR}")

Reports will be saved to: /home/riyan/Riyan/projects/pearls-aqi-predictor/data/reports/openaq_feasibility


## 2. Data Models and Custom Exceptions

A custom exception is used for OpenAQ request failures.

The `SensorCandidate` data class stores standardized metadata for every relevant pollutant sensor discovered near Karachi.

In [8]:
class OpenAQError(RuntimeError):
    """Raised when an OpenAQ request fails."""


@dataclass(frozen=True)
class SensorCandidate:
    location_id: int
    location_name: str
    provider_name: str
    is_reference_monitor: bool
    latitude: float | None
    longitude: float | None
    location_first_utc: str | None
    location_last_utc: str | None
    sensor_id: int
    sensor_name: str
    parameter_name: str
    parameter_display_name: str
    units: str

## 3. OpenAQ API Client

The API client provides:

- reusable HTTP sessions
- automatic API-key headers
- request timeouts
- retry handling
- exponential backoff
- rate-limit handling
- pagination support

In [ ]:
class OpenAQClient:
    def __init__(
        self,
        api_key: str,
        timeout_seconds: int = 30,
        max_retries: int = 3,
    ) -> None:
        self.timeout_seconds = timeout_seconds
        self.max_retries = max_retries

        self.session = requests.Session()
        self.session.headers.update(
            {
                "X-API-Key": api_key,
                "Accept": "application/json",
                "User-Agent": "karachi-aqi-feasibility-study/1.0",
            }
        )

    def get(
        self,
        path: str,
        params: dict[str, Any] | None = None,
    ) -> dict[str, Any]:
        url = f"{BASE_URL}{path}"
        last_error: Exception | None = None

        retryable_status_codes = {
            408,
            429,
            500,
            502,
            503,
            504,
        }

        for attempt in range(1, self.max_retries + 1):
            try:
                response = self.session.get(
                    url,
                    params=params,
                    timeout=self.timeout_seconds,
                )

                if response.status_code in retryable_status_codes:
                    if response.status_code == 429:
                        wait_seconds = int(
                            response.headers.get("Retry-After", "10")
                        )
                    else:
                        wait_seconds = min(2 ** attempt, 30)

                    print(
                        f"Temporary OpenAQ error "
                        f"{response.status_code}. "
                        f"Retrying in {wait_seconds}s...",
                        file=sys.stderr,
                    )

                    time.sleep(wait_seconds)
                    continue

                response.raise_for_status()

                payload = response.json()

                if not isinstance(payload, dict):
                    raise OpenAQError(
                        f"Unexpected response format from {response.url}"
                    )

                return payload

            except (
                requests.RequestException,
                ValueError,
                OpenAQError,
            ) as exc:
                last_error = exc

                if attempt < self.max_retries:
                    wait_seconds = min(2 ** attempt, 30)

                    print(
                        f"Request failed: {exc}. "
                        f"Retrying in {wait_seconds}s...",
                        file=sys.stderr,
                    )

                    time.sleep(wait_seconds)

        raise OpenAQError(
            f"OpenAQ request failed after "
            f"{self.max_retries} attempts. "
            f"URL: {url}. Last error: {last_error}"
        )


    def get_all_pages(
        self,
        path: str,
        params: dict[str, Any] | None = None,
        maximum_pages: int = 100,
    ) -> list[dict[str, Any]]:
        query = dict(params or {})
        query["limit"] = min(
            int(query.get("limit", PAGE_LIMIT)),
            PAGE_LIMIT,
        )

        all_results: list[dict[str, Any]] = []

        for page in range(1, maximum_pages + 1):
            query["page"] = page

            payload = self.get(
                path=path,
                params=query,
            )

            results = payload.get("results", [])

            if not isinstance(results, list):
                raise OpenAQError(
                    f"Unexpected results structure for endpoint {path}"
                )

            all_results.extend(results)

            meta = payload.get("meta", {}) or {}
            found = meta.get("found")

            if not results:
                break

            if isinstance(found, int) and len(all_results) >= found:
                break

            if len(results) < query["limit"]:
                break

        return all_results

In [30]:
client = OpenAQClient(api_key=API_KEY)

print("OpenAQ client initialized.")

OpenAQ client initialized.


## 4. General Utility Functions

These helper functions safely:

- extract deeply nested dictionary values
- parse timestamps
- convert timestamps to ISO strings
- calculate the longest sequence of missing hourly records

In [ ]:
def nested_value(
    value: dict[str, Any] | None,
    *keys: str,
) -> Any:
    current: Any = value

    for key in keys:
        if not isinstance(current, dict):
            return None

        current = current.get(key)

    return current


def parse_datetime(value: Any) -> pd.Timestamp | None:
    if value is None:
        return None

    if isinstance(value, dict):
        value = value.get("utc") or value.get("local")

    if not value:
        return None

    timestamp = pd.to_datetime(
        value,
        utc=True,
        errors="coerce",
    )

    if pd.isna(timestamp):
        return None

    return timestamp


def timestamp_to_string(value: Any) -> str | None:
    if value is None or pd.isna(value):
        return None

    return pd.Timestamp(value).isoformat()


def longest_consecutive_true_count(values: Any) -> int:
    longest = 0
    current = 0

    for value in values:
        if bool(value):
            current += 1
            longest = max(longest, current)
        else:
            current = 0

    return longest

## 5. Discover OpenAQ Monitoring Locations

Search for all OpenAQ monitoring locations within 25 kilometres of central Karachi.

The raw location response is retained so that the source discovery process remains reproducible and auditable.

In [13]:
def discover_locations(
    client: OpenAQClient,
) -> list[dict[str, Any]]:
    print(
        f"Searching OpenAQ locations within "
        f"{SEARCH_RADIUS_METRES / 1_000:.0f} km of central Karachi..."
    )

    locations = client.get_all_pages(
        "/locations",
        {
            "coordinates": (
                f"{KARACHI_LATITUDE},{KARACHI_LONGITUDE}"
            ),
            "radius": SEARCH_RADIUS_METRES,
            "limit": PAGE_LIMIT,
        },
    )

    print(f"Found {len(locations)} location(s).")

    return locations

In [14]:
locations = discover_locations(client)

raw_locations_path = OUTPUT_DIR / "raw_locations.json"

with raw_locations_path.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        locations,
        file,
        indent=2,
        ensure_ascii=False,
    )

print(f"Raw location metadata saved to: {raw_locations_path}")

Searching OpenAQ locations within 25 km of central Karachi...
Found 53 location(s).
Raw location metadata saved to: /home/riyan/Riyan/projects/pearls-aqi-predictor/data/reports/openaq_feasibility/raw_locations.json


## 6. Inspect Discovered Locations and Sensors

Display the following information for every location:

- location identifier
- provider
- reference-monitor status
- coordinates
- available date range
- sensor identifiers
- pollutant names
- measurement units

In [15]:
def print_location_summary(
    locations: list[dict[str, Any]],
) -> None:
    print("LOCATION AND SENSOR SUMMARY")
    print("=" * 100)

    for location in locations:
        provider = location.get("provider") or {}
        coordinates = location.get("coordinates") or {}
        sensors = location.get("sensors") or []

        print(f"\nLocation ID: {location.get('id')}")
        print(f"Name: {location.get('name')}")
        print(f"Provider: {provider.get('name')}")
        print(
            f"Reference monitor: "
            f"{bool(location.get('isMonitor'))}"
        )
        print(f"Mobile: {bool(location.get('isMobile'))}")
        print(
            "Coordinates: "
            f"{coordinates.get('latitude')}, "
            f"{coordinates.get('longitude')}"
        )
        print(
            "Available period: "
            f"{nested_value(location, 'datetimeFirst', 'utc')} "
            f"to "
            f"{nested_value(location, 'datetimeLast', 'utc')}"
        )

        if not sensors:
            print("Sensors: None")
            continue

        print("Sensors:")

        for sensor in sensors:
            parameter = sensor.get("parameter") or {}

            print(
                f"  - Sensor {sensor.get('id')}: "
                f"{parameter.get('displayName')} "
                f"({parameter.get('name')}) "
                f"[{parameter.get('units')}]"
            )

In [16]:
print_location_summary(locations)

LOCATION AND SENSOR SUMMARY

Location ID: 8156
Name: Karachi
Provider: AirNow
Reference monitor: True
Mobile: False
Coordinates: 24.8415, 67.0091
Available period: 2019-05-22T22:00:00Z to 2025-03-04T12:00:00Z
Sensors:
  - Sensor 23747: PM2.5 (pm25) [µg/m³]

Location ID: 1894633
Name: Karachi, Pakistan
Provider: Clarity
Reference monitor: False
Mobile: False
Coordinates: 24.811053632574033, 67.05825434267292
Available period: 2023-11-28T16:58:32Z to 2024-02-28T12:42:56Z
Sensors:
  - Sensor 7466368: PM1 (pm1) [µg/m³]
  - Sensor 7466369: PM10 (pm10) [µg/m³]
  - Sensor 7466370: PM2.5 (pm25) [µg/m³]
  - Sensor 7466372: Temperature (F) (temperature) [f]
  - Sensor 7466371: Temperature (C) (temperature) [c]

Location ID: 3089895
Name: Soomra Society, Scheme 33, Karachi
Provider: AirGradient
Reference monitor: False
Mobile: False
Coordinates: 24.960929, 67.161759
Available period: 2024-10-02T17:00:00Z to 2024-10-25T16:00:00Z
Sensors:
  - Sensor 10815128: PM1 (pm1) [µg/m³]
  - Sensor 10815168: 

## 7. Create the Location-Level Report

Convert raw location metadata into a structured table.

Each row represents one monitoring location and contains its:

- provider
- coordinates
- monitoring period
- available pollutants
- number of sensors

In [17]:
def create_location_report(
    locations: list[dict[str, Any]],
) -> pd.DataFrame:
    rows: list[dict[str, Any]] = []

    for location in locations:
        provider = location.get("provider") or {}
        coordinates = location.get("coordinates") or {}
        sensors = location.get("sensors") or []

        pollutant_names = sorted(
            {
                str(
                    nested_value(
                        sensor,
                        "parameter",
                        "name",
                    )
                    or ""
                ).lower()
                for sensor in sensors
                if nested_value(
                    sensor,
                    "parameter",
                    "name",
                )
            }
        )

        rows.append(
            {
                "location_id": location.get("id"),
                "name": location.get("name"),
                "provider": provider.get("name"),
                "is_reference_monitor": bool(
                    location.get("isMonitor")
                ),
                "is_mobile": bool(location.get("isMobile")),
                "latitude": coordinates.get("latitude"),
                "longitude": coordinates.get("longitude"),
                "timezone": location.get("timezone"),
                "datetime_first_utc": nested_value(
                    location,
                    "datetimeFirst",
                    "utc",
                ),
                "datetime_last_utc": nested_value(
                    location,
                    "datetimeLast",
                    "utc",
                ),
                "pollutants": ", ".join(pollutant_names),
                "sensor_count": len(sensors),
            }
        )

    return pd.DataFrame(rows)

In [18]:
location_report = create_location_report(locations)

location_report_path = OUTPUT_DIR / "locations_summary.csv"

location_report.to_csv(
    location_report_path,
    index=False,
)

display(location_report)

print(f"Location report saved to: {location_report_path}")

,location_id,name,provider,is_reference_monitor,is_mobile,latitude,longitude,timezone,datetime_first_utc,datetime_last_utc,pollutants,sensor_count
0,8156,Karachi,AirNow,True,False,24.841500,67.009100,Asia/Karachi,2019-05-22T22:00:00Z,2025-03-04T12:00:00Z,pm25,1
1,1894633,"Karachi, Pakistan",Clarity,False,False,24.811054,67.058254,Asia/Karachi,2023-11-28T16:58:32Z,2024-02-28T12:42:56Z,"pm1, pm10, pm25, temperature",5
2,3089895,"Soomra Society, Scheme 33, Karachi",AirGradient,False,False,24.960929,67.161759,Asia/Karachi,2024-10-02T17:00:00Z,2024-10-25T16:00:00Z,"pm1, pm10, pm25, relativehumidity, temperature...",6
3,3182341,"Soomra Society, Scheme 33, Karachi",AirGradient,False,False,24.960929,67.161759,Asia/Karachi,2024-10-25T18:00:00Z,2024-10-28T05:00:00Z,"pm1, pm10, pm25, relativehumidity, temperature...",6
4,3213801,"Soomra Society, Scheme 33, Karachi",AirGradient,False,False,24.960929,67.161759,Asia/Karachi,2024-11-01T19:00:00Z,2024-11-07T13:00:00Z,"pm1, pm10, pm25, relativehumidity, temperature...",6
5,4791924,Urban Resource Center,AirGradient,False,False,24.868000,67.081833,Asia/Karachi,2025-06-19T08:00:00Z,2026-07-25T05:00:00Z,"pm1, pm25, relativehumidity, temperature, um003",5
6,4793473,NED University City Campus,AirGradient,False,False,24.854021,67.014839,Asia/Karachi,2025-06-19T11:00:00Z,2026-07-14T11:00:00Z,"pm1, pm25, relativehumidity, temperature, um003",5
7,4793983,Agha Khan University IED,AirGradient,False,False,24.892054,67.070938,Asia/Karachi,2025-06-19T13:00:00Z,2026-02-16T10:00:00Z,"pm1, pm25, relativehumidity, temperature, um003",5
8,4803209,Karachi Metropolitan corporation,AirGradient,False,False,24.855514,66.925289,Asia/Karachi,2025-06-20T07:00:00Z,2025-10-14T14:00:00Z,"pm1, pm25, relativehumidity, temperature, um003",5
9,4804767,Karachi Zoo,AirGradient,False,False,24.876149,67.023053,Asia/Karachi,2025-06-20T10:00:00Z,2026-07-03T09:00:00Z,"pm1, pm25, relativehumidity, temperature, um003",5


Location report saved to: /home/riyan/Riyan/projects/pearls-aqi-predictor/data/reports/openaq_feasibility/locations_summary.csv


## 8. Extract Relevant Pollutant Sensors

Only sensors measuring the following pollutants are retained:

- PM2.5
- PM10
- nitrogen dioxide
- sulphur dioxide
- carbon monoxide
- ozone

This removes sensors that are not relevant to the AQI feasibility assessment.

In [19]:
def extract_sensor_candidates(
    locations: list[dict[str, Any]],
) -> list[SensorCandidate]:
    candidates: list[SensorCandidate] = []

    for location in locations:
        location_id = location.get("id")

        if not isinstance(location_id, int):
            continue

        coordinates = location.get("coordinates") or {}
        provider = location.get("provider") or {}

        for sensor in location.get("sensors") or []:
            parameter = sensor.get("parameter") or {}

            parameter_name = str(
                parameter.get("name") or ""
            ).lower()

            if parameter_name not in REQUIRED_POLLUTANTS:
                continue

            sensor_id = sensor.get("id")

            if not isinstance(sensor_id, int):
                continue

            candidates.append(
                SensorCandidate(
                    location_id=location_id,
                    location_name=str(
                        location.get("name")
                        or f"Location {location_id}"
                    ),
                    provider_name=str(
                        provider.get("name")
                        or "Unknown provider"
                    ),
                    is_reference_monitor=bool(
                        location.get("isMonitor")
                    ),
                    latitude=coordinates.get("latitude"),
                    longitude=coordinates.get("longitude"),
                    location_first_utc=nested_value(
                        location,
                        "datetimeFirst",
                        "utc",
                    ),
                    location_last_utc=nested_value(
                        location,
                        "datetimeLast",
                        "utc",
                    ),
                    sensor_id=sensor_id,
                    sensor_name=str(
                        sensor.get("name") or ""
                    ),
                    parameter_name=parameter_name,
                    parameter_display_name=str(
                        parameter.get("displayName")
                        or parameter_name
                    ),
                    units=str(
                        parameter.get("units") or ""
                    ),
                )
            )

    return candidates

In [ ]:
candidates = extract_sensor_candidates(locations)

candidate_table = pd.DataFrame(
    [candidate.__dict__ for candidate in candidates]
)

print(
    f"Found {len(candidates)} relevant pollutant "
    f"sensor candidate(s)."
)

display(candidate_table)

## Discovery Conclusion

The broad OpenAQ search identified multiple air-quality sensors within the Karachi search area.

Since the project target is PM2.5-based AQI forecasting, the next stage will focus only on the strongest PM2.5 candidates. Sensors were shortlisted using:

- pollutant relevance
- recent activity
- available historical period
- provider information
- location relevance
- monitoring continuity indicators

Five active PM2.5 sensors and one historical reference monitor were selected for detailed validation.

The next stage will download hourly observations only for these shortlisted sensors and evaluate their completeness, continuity, and suitability for model training and live ingestion.

In [26]:

# Strongest active Karachi PM2.5 candidates identified in test.
ACTIVE_SENSORS = {
    13406419: {
        "location_id": 4837113,
        "name": "Aga Khan University Main Campus",
        "provider": "AirGradient",
        "available_from": "2025-06-23T09:00:00Z",
    },
    13533643: {
        "location_id": 4985886,
        "name": "G3 Engineering Consultants Karachi",
        "provider": "AirGradient",
        "available_from": "2025-07-07T05:00:00Z",
    },
    13490916: {
        "location_id": 4932783,
        "name": "Climate Action Center",
        "provider": "AirGradient",
        "available_from": "2025-07-02T12:00:00Z",
    },
    13387396: {
        "location_id": 4814327,
        "name": "Zafar Memon DHA",
        "provider": "AirGradient",
        "available_from": "2025-06-21T09:00:00Z",
    },
    13368464: {
        "location_id": 4791924,
        "name": "Urban Resource Center",
        "provider": "AirGradient",
        "available_from": "2025-06-19T08:00:00Z",
    },
}

# Historical government/reference-grade monitor.
REFERENCE_SENSOR = {
    23747: {
        "location_id": 8156,
        "name": "Karachi AirNow Reference Monitor",
        "provider": "AirNow",
        "available_from": "2019-05-22T22:00:00Z",
        "available_to": "2025-03-04T12:00:00Z",
    }
}

# Evaluate active sensors over their common practical period.
ACTIVE_START = pd.Timestamp("2025-07-08T00:00:00Z")
ACTIVE_END = pd.Timestamp("2026-07-24T00:00:00Z")

print(f"Testing {len(ACTIVE_SENSORS)} active PM2.5 sensors")
print(f"Period: {ACTIVE_START} to {ACTIVE_END}")

Testing 5 active PM2.5 sensors
Period: 2025-07-08 00:00:00+00:00 to 2026-07-24 00:00:00+00:00


In [27]:
selected_sensor_ids = {
    *ACTIVE_SENSORS.keys(),
    *REFERENCE_SENSOR.keys(),
}

discovered_sensor_ids = set(
    candidate_table["sensor_id"]
    .dropna()
    .astype(int)
)

missing_sensor_ids = (
    selected_sensor_ids - discovered_sensor_ids
)

if missing_sensor_ids:
    raise ValueError(
        "The following shortlisted sensors were not found "
        f"in candidate_table: {sorted(missing_sensor_ids)}"
    )

print(
    f"Validated {len(selected_sensor_ids)} shortlisted sensors "
    "against the broad discovery results."
)

Validated 6 shortlisted sensors against the broad discovery results.


In [ ]:
def hourly_results_to_dataframe(
    results: list[dict[str, Any]],
    sensor_id: int,
    sensor_info: dict[str, Any],
) -> pd.DataFrame:
    """
    Convert OpenAQ hourly API results into a normalized DataFrame.

    The original rows are preserved so missing or invalid values can be
    measured later during sensor-quality evaluation.
    """
    rows: list[dict[str, Any]] = []

    for result in results:
        period = result.get("period") or {}
        summary = result.get("summary") or {}

        datetime_from = (
            nested_value(period, "datetimeFrom", "utc")
            or nested_value(result, "datetime", "utc")
            or nested_value(result, "date", "utc")
        )

        datetime_to = (
            nested_value(period, "datetimeTo", "utc")
            or nested_value(result, "datetimeTo", "utc")
        )

        value = (
            summary.get("avg")
            if summary.get("avg") is not None
            else result.get("value")
        )

        rows.append(
            {
                "sensor_id": sensor_id,
                "location_id": sensor_info["location_id"],
                "location_name": sensor_info["name"],
                "provider": sensor_info["provider"],
                "parameter": "pm25",
                "units": "µg/m³",
                "datetime_utc": datetime_from,
                "datetime_to_utc": datetime_to,
                "value": value,
                "minimum": summary.get("min"),
                "maximum": summary.get("max"),
                "median": summary.get("q50"),
                "standard_deviation": summary.get("sd"),
            }
        )

    dataframe = pd.DataFrame(rows)

    if dataframe.empty:
        return dataframe

    dataframe["datetime_utc"] = pd.to_datetime(
        dataframe["datetime_utc"],
        utc=True,
        errors="coerce",
    )

    dataframe["datetime_to_utc"] = pd.to_datetime(
        dataframe["datetime_to_utc"],
        utc=True,
        errors="coerce",
    )

    numeric_columns = [
        "value",
        "minimum",
        "maximum",
        "median",
        "standard_deviation",
    ]

    for column in numeric_columns:
        dataframe[column] = pd.to_numeric(
            dataframe[column],
            errors="coerce",
        )

    return dataframe

In [33]:
def fetch_hourly_chunk(
    sensor_id: int,
    sensor_info: dict[str, Any],
    start: pd.Timestamp,
    end: pd.Timestamp,
) -> pd.DataFrame:
    """
    Download and normalize one bounded period of hourly OpenAQ data.
    """
    results = client.get_all_pages(
        f"/sensors/{sensor_id}/hours",
        {
            "datetime_from": (
                start.isoformat().replace("+00:00", "Z")
            ),
            "datetime_to": (
                end.isoformat().replace("+00:00", "Z")
            ),
            "limit": PAGE_LIMIT,
        },
        maximum_pages=10,
    )

    return hourly_results_to_dataframe(
        results=results,
        sensor_id=sensor_id,
        sensor_info=sensor_info,
    )

In [35]:
def generate_chunks(
    start: pd.Timestamp,
    end: pd.Timestamp,
    days: int,
):
    """
    Generate consecutive, non-overlapping datetime chunks.
    """
    if start >= end:
        raise ValueError("start must be earlier than end.")

    if days <= 0:
        raise ValueError("days must be greater than zero.")

    current = start

    while current < end:
        chunk_end = min(
            current + pd.Timedelta(days=days),
            end,
        )

        yield current, chunk_end
        current = chunk_end


def fetch_period_safely(
    sensor_id: int,
    sensor_info: dict[str, Any],
    start: pd.Timestamp,
    end: pd.Timestamp,
    primary_chunk_days: int = 30,
    fallback_chunk_days: int = 7,
) -> pd.DataFrame:
    """
    Download a sensor's hourly data in 30-day chunks.

    Any failed primary chunk is retried using smaller 7-day chunks.
    Permanently failed periods are written to a JSON report.
    """
    collected: list[pd.DataFrame] = []
    failed_chunks: list[dict[str, str]] = []

    primary_chunks = list(
        generate_chunks(
            start=start,
            end=end,
            days=primary_chunk_days,
        )
    )

    print(
        f"\nDownloading {sensor_info['name']} "
        f"using {len(primary_chunks)} primary chunks"
    )

    for number, (chunk_start, chunk_end) in enumerate(
        primary_chunks,
        start=1,
    ):
        print(
            f"[{number}/{len(primary_chunks)}] "
            f"{chunk_start.date()} to {chunk_end.date()}"
        )

        try:
            chunk_dataframe = fetch_hourly_chunk(
                sensor_id=sensor_id,
                sensor_info=sensor_info,
                start=chunk_start,
                end=chunk_end,
            )

            print(
                f"  Returned {len(chunk_dataframe)} hourly records"
            )

            if not chunk_dataframe.empty:
                collected.append(chunk_dataframe)

        except OpenAQError as primary_error:
            print(
                f"  Primary chunk failed: {primary_error}"
            )
            print(
                f"  Retrying with "
                f"{fallback_chunk_days}-day chunks..."
            )

            fallback_chunks = list(
                generate_chunks(
                    start=chunk_start,
                    end=chunk_end,
                    days=fallback_chunk_days,
                )
            )

            for fallback_start, fallback_end in fallback_chunks:
                try:
                    fallback_dataframe = fetch_hourly_chunk(
                        sensor_id=sensor_id,
                        sensor_info=sensor_info,
                        start=fallback_start,
                        end=fallback_end,
                    )

                    print(
                        f"    {fallback_start.date()} to "
                        f"{fallback_end.date()}: "
                        f"{len(fallback_dataframe)} records"
                    )

                    if not fallback_dataframe.empty:
                        collected.append(fallback_dataframe)

                except OpenAQError as fallback_error:
                    print(
                        f"    Permanently failed: "
                        f"{fallback_start.isoformat()} to "
                        f"{fallback_end.isoformat()}"
                    )

                    failed_chunks.append(
                        {
                            "sensor_id": str(sensor_id),
                            "start": fallback_start.isoformat(),
                            "end": fallback_end.isoformat(),
                            "error": str(fallback_error),
                        }
                    )

        # Small delay between chunk requests.
        time.sleep(0.25)

    if failed_chunks:
        failed_file = (
            OUTPUT_DIR
            / f"sensor_{sensor_id}_failed_chunks.json"
        )

        with failed_file.open(
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                failed_chunks,
                file,
                indent=2,
                ensure_ascii=False,
            )

        print(f"Failed chunks saved to: {failed_file}")

    if not collected:
        return pd.DataFrame()

    combined = pd.concat(
        collected,
        ignore_index=True,
    )

    combined = (
        combined.sort_values("datetime_utc")
        .reset_index(drop=True)
    )

    return combined

In [37]:
def evaluate_sensor_period(
    dataframe: pd.DataFrame,
    sensor_id: int,
    sensor_info: dict[str, Any],
    start: pd.Timestamp,
    end: pd.Timestamp,
) -> dict[str, Any]:
    """
    Evaluate hourly completeness and basic PM2.5 data quality
    for one selected sensor over a fixed period.
    """
    if start >= end:
        raise ValueError("start must be earlier than end.")

    expected_index = pd.date_range(
        start=start.floor("h"),
        end=end.floor("h"),
        freq="h",
        inclusive="left",
    )

    expected_hours = len(expected_index)

    base_report = {
        "sensor_id": sensor_id,
        "location_id": sensor_info["location_id"],
        "location_name": sensor_info["name"],
        "provider": sensor_info["provider"],
        "period_start": start.isoformat(),
        "period_end": end.isoformat(),
        "expected_hours": expected_hours,
    }

    if dataframe.empty:
        return {
            **base_report,
            "raw_records": 0,
            "observed_unique_hours": 0,
            "missing_hours": expected_hours,
            "completeness_percent": 0.0,
            "longest_missing_gap_hours": expected_hours,
            "duplicate_hour_rows": 0,
            "invalid_timestamp_rows": 0,
            "missing_value_rows": 0,
            "negative_values": 0,
            "zero_values": 0,
            "minimum_pm25": None,
            "maximum_pm25": None,
            "mean_pm25": None,
            "median_pm25": None,
            "first_observation": None,
            "last_observation": None,
        }

    required_columns = {
        "sensor_id",
        "datetime_utc",
        "value",
    }

    missing_columns = required_columns - set(dataframe.columns)

    if missing_columns:
        raise ValueError(
            "Sensor DataFrame is missing required columns: "
            f"{sorted(missing_columns)}"
        )

    raw = dataframe.copy()

    invalid_timestamp_rows = int(
        raw["datetime_utc"].isna().sum()
    )

    missing_value_rows = int(
        raw["value"].isna().sum()
    )

    clean = raw.dropna(
        subset=["datetime_utc", "value"]
    ).copy()

    clean["hour_utc"] = (
        clean["datetime_utc"].dt.floor("h")
    )

    duplicate_hour_rows = int(
        clean.duplicated(
            subset=["sensor_id", "hour_utc"],
            keep=False,
        ).sum()
    )

    hourly = (
        clean.sort_values("datetime_utc")
        .drop_duplicates(
            subset=["sensor_id", "hour_utc"],
            keep="last",
        )
        .set_index("hour_utc")
    )

    aligned = hourly.reindex(expected_index)

    present_mask = aligned["value"].notna()
    missing_mask = ~present_mask

    observed_hours = int(present_mask.sum())
    missing_hours = expected_hours - observed_hours

    completeness_percent = (
        round(
            observed_hours / expected_hours * 100,
            2,
        )
        if expected_hours
        else 0.0
    )

    valid_values = clean["value"]

    first_observation = (
        clean["datetime_utc"].min()
        if not clean.empty
        else None
    )

    last_observation = (
        clean["datetime_utc"].max()
        if not clean.empty
        else None
    )

    return {
        **base_report,
        "raw_records": len(raw),
        "observed_unique_hours": observed_hours,
        "missing_hours": missing_hours,
        "completeness_percent": completeness_percent,
        "longest_missing_gap_hours": (
            longest_consecutive_true_count(missing_mask)
        ),
        "duplicate_hour_rows": duplicate_hour_rows,
        "invalid_timestamp_rows": invalid_timestamp_rows,
        "missing_value_rows": missing_value_rows,
        "negative_values": int(
            (valid_values < 0).sum()
        ),
        "zero_values": int(
            (valid_values == 0).sum()
        ),
        "minimum_pm25": (
            round(float(valid_values.min()), 3)
            if not valid_values.empty
            else None
        ),
        "maximum_pm25": (
            round(float(valid_values.max()), 3)
            if not valid_values.empty
            else None
        ),
        "mean_pm25": (
            round(float(valid_values.mean()), 3)
            if not valid_values.empty
            else None
        ),
        "median_pm25": (
            round(float(valid_values.median()), 3)
            if not valid_values.empty
            else None
        ),
        "first_observation": (
            first_observation.isoformat()
            if first_observation is not None
            else None
        ),
        "last_observation": (
            last_observation.isoformat()
            if last_observation is not None
            else None
        ),
    }

In [38]:
TEST_SENSOR_ID = 13406419
TEST_SENSOR_INFO = ACTIVE_SENSORS[TEST_SENSOR_ID]

test_start = pd.Timestamp("2026-06-24T00:00:00Z")
test_end = pd.Timestamp("2026-07-24T00:00:00Z")

test_df = fetch_period_safely(
    sensor_id=TEST_SENSOR_ID,
    sensor_info=TEST_SENSOR_INFO,
    start=test_start,
    end=test_end,
)

print("\nTest result")
print(f"Rows: {len(test_df)}")
print(f"First timestamp: {test_df['datetime_utc'].min()}")
print(f"Last timestamp: {test_df['datetime_utc'].max()}")

display(test_df.head())
display(test_df.tail())


[1/1] 2026-06-24 to 2026-07-24
  Returned 720 hourly records

Test result
Rows: 720
First timestamp: 2026-06-24 00:00:00+00:00
Last timestamp: 2026-07-23 23:00:00+00:00


,sensor_id,location_id,location_name,provider,parameter,units,datetime_utc,datetime_to_utc,value,minimum,maximum,median,standard_deviation
0,13406419,4837113,Aga Khan University Main Campus,AirGradient,pm25,µg/m³,2026-06-24 00:00:00+00:00,2026-06-24 01:00:00+00:00,20.9,20.9,20.9,NaN,NaN
1,13406419,4837113,Aga Khan University Main Campus,AirGradient,pm25,µg/m³,2026-06-24 01:00:00+00:00,2026-06-24 02:00:00+00:00,20.2,20.2,20.2,NaN,NaN
2,13406419,4837113,Aga Khan University Main Campus,AirGradient,pm25,µg/m³,2026-06-24 02:00:00+00:00,2026-06-24 03:00:00+00:00,19.6,19.6,19.6,NaN,NaN
3,13406419,4837113,Aga Khan University Main Campus,AirGradient,pm25,µg/m³,2026-06-24 03:00:00+00:00,2026-06-24 04:00:00+00:00,17.8,17.8,17.8,NaN,NaN
4,13406419,4837113,Aga Khan University Main Campus,AirGradient,pm25,µg/m³,2026-06-24 04:00:00+00:00,2026-06-24 05:00:00+00:00,21.3,21.3,21.3,NaN,NaN


,sensor_id,location_id,location_name,provider,parameter,units,datetime_utc,datetime_to_utc,value,minimum,maximum,median,standard_deviation
715,13406419,4837113,Aga Khan University Main Campus,AirGradient,pm25,µg/m³,2026-07-23 19:00:00+00:00,2026-07-23 20:00:00+00:00,21.2,21.2,21.2,NaN,NaN
716,13406419,4837113,Aga Khan University Main Campus,AirGradient,pm25,µg/m³,2026-07-23 20:00:00+00:00,2026-07-23 21:00:00+00:00,32.5,32.5,32.5,NaN,NaN
717,13406419,4837113,Aga Khan University Main Campus,AirGradient,pm25,µg/m³,2026-07-23 21:00:00+00:00,2026-07-23 22:00:00+00:00,36.9,36.9,36.9,NaN,NaN
718,13406419,4837113,Aga Khan University Main Campus,AirGradient,pm25,µg/m³,2026-07-23 22:00:00+00:00,2026-07-23 23:00:00+00:00,42.5,42.5,42.5,NaN,NaN
719,13406419,4837113,Aga Khan University Main Campus,AirGradient,pm25,µg/m³,2026-07-23 23:00:00+00:00,2026-07-24 00:00:00+00:00,24.8,24.8,24.8,NaN,NaN


In [10]:
def longest_missing_run(missing_mask: pd.Series) -> int:
    longest = 0
    current = 0

    for missing in missing_mask:
        if bool(missing):
            current += 1
            longest = max(longest, current)
        else:
            current = 0

    return longest


def evaluate_sensor_period(
    dataframe: pd.DataFrame,
    sensor_id: int,
    sensor_info: dict[str, Any],
    start: pd.Timestamp,
    end: pd.Timestamp,
) -> dict[str, Any]:
    expected_index = pd.date_range(
        start=start.floor("h"),
        end=end.floor("h"),
        freq="h",
        inclusive="left",
    )

    if dataframe.empty:
        return {
            "sensor_id": sensor_id,
            "location_name": sensor_info["name"],
            "expected_hours": len(expected_index),
            "observed_hours": 0,
            "completeness_percent": 0.0,
            "longest_missing_gap_hours": len(expected_index),
        }

    clean = dataframe.copy()

    clean["hour_utc"] = clean["datetime_utc"].dt.floor("h")

    duplicate_rows = int(
        clean.duplicated(
            subset=["sensor_id", "hour_utc"],
            keep=False,
        ).sum()
    )

    hourly = (
        clean.sort_values("datetime_utc")
        .drop_duplicates(
            subset=["sensor_id", "hour_utc"],
            keep="last",
        )
        .set_index("hour_utc")
    )

    aligned = hourly.reindex(expected_index)

    present_mask = aligned["value"].notna()
    missing_mask = ~present_mask

    expected_hours = len(expected_index)
    observed_hours = int(present_mask.sum())

    completeness = (
        round(observed_hours / expected_hours * 100, 2)
        if expected_hours
        else 0.0
    )

    valid_values = clean["value"].dropna()

    return {
        "sensor_id": sensor_id,
        "location_id": sensor_info["location_id"],
        "location_name": sensor_info["name"],
        "provider": sensor_info["provider"],
        "period_start": start.isoformat(),
        "period_end": end.isoformat(),
        "expected_hours": expected_hours,
        "raw_records": len(dataframe),
        "observed_unique_hours": observed_hours,
        "missing_hours": expected_hours - observed_hours,
        "completeness_percent": completeness,
        "longest_missing_gap_hours": longest_missing_run(
            missing_mask
        ),
        "duplicate_hour_rows": duplicate_rows,
        "negative_values": int((valid_values < 0).sum()),
        "zero_values": int((valid_values == 0).sum()),
        "minimum_pm25": (
            round(float(valid_values.min()), 3)
            if not valid_values.empty
            else None
        ),
        "maximum_pm25": (
            round(float(valid_values.max()), 3)
            if not valid_values.empty
            else None
        ),
        "mean_pm25": (
            round(float(valid_values.mean()), 3)
            if not valid_values.empty
            else None
        ),
        "median_pm25": (
            round(float(valid_values.median()), 3)
            if not valid_values.empty
            else None
        ),
        "first_observation": (
            dataframe["datetime_utc"].min().isoformat()
            if not dataframe.empty
            else None
        ),
        "last_observation": (
            dataframe["datetime_utc"].max().isoformat()
            if not dataframe.empty
            else None
        ),
    }


test_report = evaluate_sensor_period(
    dataframe=test_df,
    sensor_id=TEST_SENSOR_ID,
    sensor_info=TEST_SENSOR_INFO,
    start=test_start,
    end=test_end,
)

display(pd.DataFrame([test_report]))

,sensor_id,location_id,location_name,provider,period_start,period_end,expected_hours,raw_records,observed_unique_hours,missing_hours,...,longest_missing_gap_hours,duplicate_hour_rows,negative_values,zero_values,minimum_pm25,maximum_pm25,mean_pm25,median_pm25,first_observation,last_observation
0,13406419,4837113,Aga Khan University Main Campus,AirGradient,2026-06-24T00:00:00+00:00,2026-07-24T00:00:00+00:00,720,720,720,0,...,0,0,0,0,8.0,54.4,23.759,23.3,2026-06-24T00:00:00+00:00,2026-07-23T23:00:00+00:00


In [39]:
all_sensor_data: dict[int, pd.DataFrame] = {}
evaluation_rows: list[dict[str, Any]] = []

for sensor_id, sensor_info in ACTIVE_SENSORS.items():
    sensor_start = max(
        ACTIVE_START,
        pd.Timestamp(sensor_info["available_from"]),
    )
    sensor_end = ACTIVE_END

    print(
        f"\n{'=' * 80}\n"
        f"Processing sensor {sensor_id}: {sensor_info['name']}\n"
        f"Period: {sensor_start} to {sensor_end}"
    )

    try:
        dataframe = fetch_period_safely(
            sensor_id=sensor_id,
            sensor_info=sensor_info,
            start=sensor_start,
            end=sensor_end,
            primary_chunk_days=30,
            fallback_chunk_days=7,
        )

        if not dataframe.empty:
            dataframe = (
                dataframe.sort_values("datetime_utc")
                .reset_index(drop=True)
            )

        all_sensor_data[sensor_id] = dataframe

        evaluation = evaluate_sensor_period(
            dataframe=dataframe,
            sensor_id=sensor_id,
            sensor_info=sensor_info,
            start=sensor_start,
            end=sensor_end,
        )

        evaluation_rows.append(evaluation)

        if dataframe.empty:
            print(
                f"No usable hourly data returned for sensor {sensor_id}. "
                "CSV file was not created."
            )
            continue

        start_label = sensor_start.strftime("%Y%m%d")
        end_label = sensor_end.strftime("%Y%m%d")

        output_file = (
            OUTPUT_DIR
            / (
                f"sensor_{sensor_id}_pm25_"
                f"{start_label}_{end_label}.csv"
            )
        )

        dataframe.to_csv(
            output_file,
            index=False,
        )

        print(f"Saved: {output_file}")

    except OpenAQError as exc:
        print(
            f"Sensor {sensor_id} failed: {exc}",
            file=sys.stderr,
        )

        all_sensor_data[sensor_id] = pd.DataFrame()

        evaluation_rows.append(
            {
                "sensor_id": sensor_id,
                "location_id": sensor_info["location_id"],
                "location_name": sensor_info["name"],
                "provider": sensor_info["provider"],
                "period_start": sensor_start.isoformat(),
                "period_end": sensor_end.isoformat(),
                "expected_hours": len(
                    pd.date_range(
                        start=sensor_start.floor("h"),
                        end=sensor_end.floor("h"),
                        freq="h",
                        inclusive="left",
                    )
                ),
                "raw_records": 0,
                "observed_unique_hours": 0,
                "missing_hours": None,
                "completeness_percent": 0.0,
                "longest_missing_gap_hours": None,
                "error": str(exc),
            }
        )


Processing sensor 13406419: Aga Khan University Main Campus
Period: 2025-07-08 00:00:00+00:00 to 2026-07-24 00:00:00+00:00

[1/13] 2025-07-08 to 2025-08-07
  Returned 720 hourly records
[2/13] 2025-08-07 to 2025-09-06
  Returned 716 hourly records
[3/13] 2025-09-06 to 2025-10-06
  Returned 707 hourly records
[4/13] 2025-10-06 to 2025-11-05
  Returned 315 hourly records
[5/13] 2025-11-05 to 2025-12-05
  Returned 162 hourly records
[6/13] 2025-12-05 to 2026-01-04
  Returned 717 hourly records
[7/13] 2026-01-04 to 2026-02-03
  Returned 382 hourly records
[8/13] 2026-02-03 to 2026-03-05
  Returned 194 hourly records
[9/13] 2026-03-05 to 2026-04-04
  Returned 720 hourly records
[10/13] 2026-04-04 to 2026-05-04
  Returned 716 hourly records
[11/13] 2026-05-04 to 2026-06-03
  Returned 714 hourly records
[12/13] 2026-06-03 to 2026-07-03
  Returned 720 hourly records
[13/13] 2026-07-03 to 2026-07-24
  Returned 504 hourly records
Saved: /home/riyan/Riyan/projects/pearls-aqi-predictor/data/repor

In [40]:
active_sensor_report = pd.DataFrame(evaluation_rows)

active_sensor_report = active_sensor_report.sort_values(
    by="completeness_percent",
    ascending=False,
    na_position="last",
).reset_index(drop=True)

display(active_sensor_report)

report_file = OUTPUT_DIR / "active_sensor_evaluation_report.csv"

active_sensor_report.to_csv(
    report_file,
    index=False,
)

print(f"Evaluation report saved to: {report_file}")

,sensor_id,location_id,location_name,provider,period_start,period_end,expected_hours,raw_records,observed_unique_hours,missing_hours,...,invalid_timestamp_rows,missing_value_rows,negative_values,zero_values,minimum_pm25,maximum_pm25,mean_pm25,median_pm25,first_observation,last_observation
0,13387396,4814327,Zafar Memon DHA,AirGradient,2025-07-08T00:00:00+00:00,2026-07-24T00:00:00+00:00,9144,8695,8695,449,...,0,0,0,5,0.000,533.120,35.706,20.020,2025-07-08T00:00:00+00:00,2026-07-23T23:00:00+00:00
1,13490916,4932783,Climate Action Center,AirGradient,2025-07-08T00:00:00+00:00,2026-07-24T00:00:00+00:00,9144,7524,7524,1620,...,0,0,0,0,0.297,328.763,46.516,34.600,2025-07-08T00:00:00+00:00,2026-07-23T23:00:00+00:00
2,13368464,4791924,Urban Resource Center,AirGradient,2025-07-08T00:00:00+00:00,2026-07-24T00:00:00+00:00,9144,7482,7482,1662,...,0,0,0,0,2.100,535.369,52.772,40.985,2025-07-08T00:00:00+00:00,2026-07-23T23:00:00+00:00
3,13406419,4837113,Aga Khan University Main Campus,AirGradient,2025-07-08T00:00:00+00:00,2026-07-24T00:00:00+00:00,9144,7287,7287,1857,...,0,0,0,0,0.484,351.791,38.838,27.800,2025-07-08T00:00:00+00:00,2026-07-23T23:00:00+00:00
4,13533643,4985886,G3 Engineering Consultants Karachi,AirGradient,2025-07-08T00:00:00+00:00,2026-07-24T00:00:00+00:00,9144,6779,6779,2365,...,0,0,0,0,0.100,462.094,42.924,25.711,2025-07-08T00:00:00+00:00,2026-07-23T23:00:00+00:00


Evaluation report saved to: /home/riyan/Riyan/projects/pearls-aqi-predictor/data/reports/openaq_feasibility/active_sensor_evaluation_report.csv


In [41]:
BEST_SENSOR_ID = 13387396

if BEST_SENSOR_ID not in all_sensor_data:
    raise KeyError(
        f"Sensor {BEST_SENSOR_ID} was not found in all_sensor_data."
    )

best_df = all_sensor_data[BEST_SENSOR_ID].copy()

if best_df.empty:
    raise ValueError(
        f"Sensor {BEST_SENSOR_ID} has no downloaded hourly data."
    )

required_columns = {
    "datetime_utc",
    "value",
}

missing_columns = required_columns - set(best_df.columns)

if missing_columns:
    raise ValueError(
        "Best sensor DataFrame is missing required columns: "
        f"{sorted(missing_columns)}"
    )

best_sensor_info = ACTIVE_SENSORS[BEST_SENSOR_ID]

best_start = max(
    ACTIVE_START,
    pd.Timestamp(best_sensor_info["available_from"]),
)

expected_index = pd.date_range(
    start=best_start.floor("h"),
    end=ACTIVE_END.floor("h"),
    freq="h",
    inclusive="left",
)

best_hourly = (
    best_df.dropna(subset=["datetime_utc"])
    .assign(
        hour_utc=lambda dataframe: (
            dataframe["datetime_utc"].dt.floor("h")
        )
    )
    .sort_values("datetime_utc")
    .drop_duplicates(
        subset=["hour_utc"],
        keep="last",
    )
    .set_index("hour_utc")
    .reindex(expected_index)
)

best_hourly.index.name = "datetime_utc"
best_hourly["is_missing"] = best_hourly["value"].isna()

# Start a new group whenever the missing/non-missing state changes.
best_hourly["gap_group"] = (
    best_hourly["is_missing"]
    .ne(best_hourly["is_missing"].shift())
    .cumsum()
)

missing_rows = best_hourly[
    best_hourly["is_missing"]
].copy()

if missing_rows.empty:
    missing_periods = pd.DataFrame(
        columns=[
            "gap_start",
            "gap_end",
            "missing_hours",
        ]
    )
else:
    missing_periods = (
        missing_rows.groupby("gap_group")
        .agg(
            gap_start=("is_missing", lambda series: series.index.min()),
            gap_end=("is_missing", lambda series: series.index.max()),
            missing_hours=("is_missing", "size"),
        )
        .reset_index(drop=True)
        .sort_values(
            "missing_hours",
            ascending=False,
        )
        .reset_index(drop=True)
    )

display(missing_periods.head(20))

,gap_start,gap_end,missing_hours
0,2026-04-03 09:00:00+00:00,2026-04-14 18:00:00+00:00,274
1,2026-01-21 22:00:00+00:00,2026-01-22 14:00:00+00:00,17
2,2026-07-13 14:00:00+00:00,2026-07-14 02:00:00+00:00,13
3,2026-05-31 03:00:00+00:00,2026-05-31 14:00:00+00:00,12
4,2026-05-09 07:00:00+00:00,2026-05-09 18:00:00+00:00,12
5,2025-10-04 19:00:00+00:00,2025-10-05 04:00:00+00:00,10
6,2025-11-14 05:00:00+00:00,2025-11-14 12:00:00+00:00,8
7,2025-10-20 12:00:00+00:00,2025-10-20 19:00:00+00:00,8
8,2025-11-10 23:00:00+00:00,2025-11-11 05:00:00+00:00,7
9,2025-09-28 08:00:00+00:00,2025-09-28 14:00:00+00:00,7


In [42]:
missing_periods_file = (
    OUTPUT_DIR
    / f"sensor_{BEST_SENSOR_ID}_missing_periods.csv"
)

missing_periods.to_csv(
    missing_periods_file,
    index=False,
)

print(f"Missing-period report saved to: {missing_periods_file}")

Missing-period report saved to: /home/riyan/Riyan/projects/pearls-aqi-predictor/data/reports/openaq_feasibility/sensor_13387396_missing_periods.csv


In [43]:
REFERENCE_SENSOR_ID = 23747
REFERENCE_INFO = REFERENCE_SENSOR[REFERENCE_SENSOR_ID]

# Historical-only evaluation period for the AirNow reference monitor.
reference_start = pd.Timestamp("2024-01-01T00:00:00Z")
reference_end = pd.Timestamp("2025-03-04T13:00:00Z")

print(
    f"Downloading historical reference sensor "
    f"{REFERENCE_SENSOR_ID}: {REFERENCE_INFO['name']}"
)
print(f"Period: {reference_start} to {reference_end}")

reference_df = fetch_period_safely(
    sensor_id=REFERENCE_SENSOR_ID,
    sensor_info=REFERENCE_INFO,
    start=reference_start,
    end=reference_end,
    primary_chunk_days=30,
    fallback_chunk_days=7,
)

if not reference_df.empty:
    reference_df = (
        reference_df.sort_values("datetime_utc")
        .reset_index(drop=True)
    )

    reference_output_file = (
        OUTPUT_DIR
        / "airnow_reference_pm25_20240101_20250304.csv"
    )

    reference_df.to_csv(
        reference_output_file,
        index=False,
    )

    print(f"Reference data saved to: {reference_output_file}")
else:
    print(
        "No hourly records were returned for the "
        "historical reference sensor."
    )

reference_report = evaluate_sensor_period(
    dataframe=reference_df,
    sensor_id=REFERENCE_SENSOR_ID,
    sensor_info=REFERENCE_INFO,
    start=reference_start,
    end=reference_end,
)

display(pd.DataFrame([reference_report]))

Period: 2024-01-01 00:00:00+00:00 to 2025-03-04 13:00:00+00:00

[1/15] 2024-01-01 to 2024-01-31
  Returned 701 hourly records
[2/15] 2024-01-31 to 2024-03-01
  Returned 615 hourly records
[3/15] 2024-03-01 to 2024-03-31
  Returned 710 hourly records
[4/15] 2024-03-31 to 2024-04-30
  Returned 720 hourly records
[5/15] 2024-04-30 to 2024-05-30
  Returned 720 hourly records
[6/15] 2024-05-30 to 2024-06-29
  Returned 720 hourly records
[7/15] 2024-06-29 to 2024-07-29
  Returned 720 hourly records
[8/15] 2024-07-29 to 2024-08-28
  Returned 720 hourly records
[9/15] 2024-08-28 to 2024-09-27
  Returned 720 hourly records
[10/15] 2024-09-27 to 2024-10-27
  Returned 720 hourly records
[11/15] 2024-10-27 to 2024-11-26
  Returned 720 hourly records
[12/15] 2024-11-26 to 2024-12-26
  Returned 720 hourly records
[13/15] 2024-12-26 to 2025-01-25
  Returned 720 hourly records
[14/15] 2025-01-25 to 2025-02-24
  Returned 720 hourly records
[15/15] 2025-02-24 to 2025-03-04
  Returned 204 hourly records


,sensor_id,location_id,location_name,provider,period_start,period_end,expected_hours,raw_records,observed_unique_hours,missing_hours,...,invalid_timestamp_rows,missing_value_rows,negative_values,zero_values,minimum_pm25,maximum_pm25,mean_pm25,median_pm25,first_observation,last_observation
0,23747,8156,Karachi AirNow Reference Monitor,AirNow,2024-01-01T00:00:00+00:00,2025-03-04T13:00:00+00:00,10285,10150,7199,3086,...,0,2951,0,4,0.0,897.0,49.934,33.0,2024-01-01T00:00:00+00:00,2025-03-04T11:00:00+00:00


In [45]:
if BEST_SENSOR_ID not in all_sensor_data:
    raise KeyError(
        f"Sensor {BEST_SENSOR_ID} was not found in all_sensor_data."
    )

best_df = all_sensor_data[BEST_SENSOR_ID].copy()

if best_df.empty:
    raise ValueError(
        f"Sensor {BEST_SENSOR_ID} has no downloaded hourly data."
    )

required_columns = {
    "datetime_utc",
    "value",
    "minimum",
    "maximum",
}

missing_columns = required_columns - set(best_df.columns)

if missing_columns:
    raise ValueError(
        "Best sensor DataFrame is missing required columns: "
        f"{sorted(missing_columns)}"
    )

best_df = (
    best_df.sort_values("datetime_utc")
    .reset_index(drop=True)
)

zero_rows = best_df[
    best_df["value"].eq(0)
].copy()

print(f"Zero-reading count: {len(zero_rows)}")

if zero_rows.empty:
    print("No zero PM2.5 readings were found.")
else:
    display(zero_rows)

    for row_index in zero_rows.index:
        print("\n" + "=" * 80)
        print(
            "Context around zero reading at "
            f"{best_df.loc[row_index, 'datetime_utc']}"
        )

        context_start = max(0, row_index - 5)
        context_end = min(
            len(best_df),
            row_index + 6,
        )

        context = best_df.iloc[
            context_start:context_end
        ][
            [
                "datetime_utc",
                "value",
                "minimum",
                "maximum",
            ]
        ]

        display(context)

Zero-reading count: 5


,sensor_id,location_id,location_name,provider,parameter,units,datetime_utc,datetime_to_utc,value,minimum,maximum,median,standard_deviation
6198,13387396,4814327,Zafar Memon DHA,AirGradient,pm25,µg/m³,2026-03-28 03:00:00+00:00,2026-03-28 04:00:00+00:00,0.0,0.0,0.0,NaN,NaN
6200,13387396,4814327,Zafar Memon DHA,AirGradient,pm25,µg/m³,2026-03-28 05:00:00+00:00,2026-03-28 06:00:00+00:00,0.0,0.0,0.0,NaN,NaN
6206,13387396,4814327,Zafar Memon DHA,AirGradient,pm25,µg/m³,2026-03-28 11:00:00+00:00,2026-03-28 12:00:00+00:00,0.0,0.0,0.0,NaN,NaN
6207,13387396,4814327,Zafar Memon DHA,AirGradient,pm25,µg/m³,2026-03-28 12:00:00+00:00,2026-03-28 13:00:00+00:00,0.0,0.0,0.0,NaN,NaN
6229,13387396,4814327,Zafar Memon DHA,AirGradient,pm25,µg/m³,2026-03-29 10:00:00+00:00,2026-03-29 11:00:00+00:00,0.0,0.0,0.0,NaN,NaN



Context around zero reading at 2026-03-28 03:00:00+00:00


,datetime_utc,value,minimum,maximum
6193,2026-03-27 22:00:00+00:00,0.1,0.1,0.1
6194,2026-03-27 23:00:00+00:00,0.1,0.1,0.1
6195,2026-03-28 00:00:00+00:00,0.3,0.3,0.3
6196,2026-03-28 01:00:00+00:00,0.5,0.5,0.5
6197,2026-03-28 02:00:00+00:00,0.2,0.2,0.2
6198,2026-03-28 03:00:00+00:00,0.0,0.0,0.0
6199,2026-03-28 04:00:00+00:00,0.4,0.4,0.4
6200,2026-03-28 05:00:00+00:00,0.0,0.0,0.0
6201,2026-03-28 06:00:00+00:00,0.1,0.1,0.1
6202,2026-03-28 07:00:00+00:00,0.2,0.2,0.2



Context around zero reading at 2026-03-28 05:00:00+00:00


...

,datetime_utc,value,minimum,maximum
6201,2026-03-28 06:00:00+00:00,0.1,0.1,0.1
6202,2026-03-28 07:00:00+00:00,0.2,0.2,0.2
6203,2026-03-28 08:00:00+00:00,5.1,5.1,5.1
6204,2026-03-28 09:00:00+00:00,1.7,1.7,1.7
6205,2026-03-28 10:00:00+00:00,0.2,0.2,0.2
6206,2026-03-28 11:00:00+00:00,0.0,0.0,0.0
6207,2026-03-28 12:00:00+00:00,0.0,0.0,0.0
6208,2026-03-28 13:00:00+00:00,0.3,0.3,0.3
6209,2026-03-28 14:00:00+00:00,0.5,0.5,0.5
6210,2026-03-28 15:00:00+00:00,0.7,0.7,0.7



Context around zero reading at 2026-03-28 12:00:00+00:00


,datetime_utc,value,minimum,maximum
6202,2026-03-28 07:00:00+00:00,0.2,0.2,0.2
6203,2026-03-28 08:00:00+00:00,5.1,5.1,5.1
6204,2026-03-28 09:00:00+00:00,1.7,1.7,1.7
6205,2026-03-28 10:00:00+00:00,0.2,0.2,0.2
6206,2026-03-28 11:00:00+00:00,0.0,0.0,0.0
6207,2026-03-28 12:00:00+00:00,0.0,0.0,0.0
6208,2026-03-28 13:00:00+00:00,0.3,0.3,0.3
6209,2026-03-28 14:00:00+00:00,0.5,0.5,0.5
6210,2026-03-28 15:00:00+00:00,0.7,0.7,0.7
6211,2026-03-28 16:00:00+00:00,1.5,1.5,1.5



Context around zero reading at 2026-03-29 10:00:00+00:00


,datetime_utc,value,minimum,maximum
6224,2026-03-29 05:00:00+00:00,1.6,1.6,1.6
6225,2026-03-29 06:00:00+00:00,3.9,3.9,3.9
6226,2026-03-29 07:00:00+00:00,1.8,1.8,1.8
6227,2026-03-29 08:00:00+00:00,0.8,0.8,0.8
6228,2026-03-29 09:00:00+00:00,0.6,0.6,0.6
6229,2026-03-29 10:00:00+00:00,0.0,0.0,0.0
6230,2026-03-29 11:00:00+00:00,0.1,0.1,0.1
6231,2026-03-29 12:00:00+00:00,0.3,0.3,0.3
6232,2026-03-29 13:00:00+00:00,1.7,1.7,1.7
6233,2026-03-29 14:00:00+00:00,2.9,2.9,2.9


## Final OpenAQ source decision

Selected source:

- Location: Zafar Memon DHA
- Location ID: 4814327
- Sensor ID: 13387396
- Provider: AirGradient
- Parameter: PM2.5
- Unit: µg/m³

The selected sensor provides the strongest overall balance of historical
coverage, recent availability, and suitability for the project location.

Known limitations:

- 449 missing source hours
- longest outage of 274 hours
- five exact-zero readings requiring treatment as missing
- high values retained because their surrounding temporal patterns appear plausible